In [9]:
!pwd

/Users/davidbird/github-work/quarto-blog/notebooks


In [12]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

/Users/davidbird/github-work/quarto-blog


In [ ]:
!head -5 src/waste_operators/load.py

In [ ]:
!cp ~/Desktop/load.py src/waste_operators/load.py

In [ ]:
!pip install pyxlsb
!ls data/raw

In [8]:
%run src/waste_operators/load.py

Exception: File `'src/waste_operators/load.py'` not found.

In [6]:
import duckdb
duckdb.sql("SELECT operator, SUM(tonnes_received) AS tonnes FROM 'data/interim/waste_received_2025.parquet' GROUP BY 1 ORDER BY 2 DESC LIMIT 10").df()

IOException: IO Error: No files found that match the pattern "data/interim/waste_received_2025.parquet"

In [7]:
import duckdb
duckdb.sql("""
SELECT facility_type, COUNT(DISTINCT epr_number) AS permits,
       ROUND(SUM(tonnes_received)/1e6, 1) AS million_tonnes
FROM 'data/interim/waste_received_2025.parquet'
GROUP BY 1 ORDER BY 3 DESC LIMIT 10
""").df()

IOException: IO Error: No files found that match the pattern "data/interim/waste_received_2025.parquet"

In [ ]:
duckdb.sql(r"""
WITH ops AS (SELECT DISTINCT operator FROM 'data/interim/waste_received_2025.parquet'),
n AS (
  SELECT operator,
         regexp_replace(
           regexp_replace(regexp_replace(lower(operator), '&', ' and '), '[^a-z0-9 ]', ' ', 'g'),
           '\b(limited|ltd|plc|llp)\b', ' ', 'g') AS stem
  FROM ops)
SELECT COUNT(*) AS raw_names,
       COUNT(DISTINCT trim(regexp_replace(stem, '\s+', ' ', 'g'))) AS standardised_names
FROM n
""").df()

In [13]:
f = "'data/interim/waste_received_2025.parquet'"
q = lambda sql: duckdb.sql(sql).df()

# 1. What is Thames Water's tonnage?
q(f"""SELECT ewc_chapter, facility_type, ROUND(SUM(tonnes_received)/1e6,1) AS mt
FROM {f} WHERE operator = 'Thames Water Utilities Limited'
GROUP BY 1,2 ORDER BY 3 DESC LIMIT 8""")


,ewc_chapter,facility_type,mt
0,19 - WASTE AND WATER TREATMENT WASTES,Anaerobic digestion,16.3
1,19 - WASTE AND WATER TREATMENT WASTES,Physical-Chemical Treatment,7.0
2,19 - WASTE AND WATER TREATMENT WASTES,Combustion,5.4
3,19 - WASTE AND WATER TREATMENT WASTES,Biological Treatment,2.3
4,19 - WASTE AND WATER TREATMENT WASTES,Recovery of Waste,0.9
5,16 - WASTES NOT OTHERWISE SPECIFIED IN THE LIST,Recovery of Waste,0.0
6,19 - WASTE AND WATER TREATMENT WASTES,Sewage Sludge Incinerator,0.0
7,16 - WASTES NOT OTHERWISE SPECIFIED IN THE LIST,Biological Treatment,0.0


In [ ]:
# 2. Biggest waste chapters overall
q(f"""SELECT ewc_chapter, ROUND(SUM(tonnes_received)/1e6,1) AS mt,
       COUNT(DISTINCT operator) AS operators
FROM {f} GROUP BY 1 ORDER BY 2 DESC LIMIT 10""")


In [ ]:
# 3. What does 'fate' contain?
q(f"""SELECT fate, ROUND(SUM(tonnes_received)/1e6,1) AS mt
FROM {f} GROUP BY 1 ORDER BY 2 DESC""")


In [ ]:
# 4. How fragmented are the big groups? (first word of the name is a rough proxy)
q(f"""SELECT regexp_extract(lower(operator), '^[a-z0-9]+', 0) AS first_word,
       COUNT(DISTINCT operator) AS names, ROUND(SUM(tonnes_received)/1e6,1) AS mt
FROM {f} GROUP BY 1 HAVING COUNT(DISTINCT operator) > 1 ORDER BY 3 DESC LIMIT 15""")

In [ ]:
q(f"""SELECT ewc_sub_chapter, ROUND(SUM(tonnes_received)/1e6,1) AS mt,
       COUNT(DISTINCT operator) AS operators
FROM {f} WHERE ewc_chapter LIKE '19%' GROUP BY 1 ORDER BY 2 DESC LIMIT 8""")

In [ ]:
import sys; sys.path.append("src")
from waste_operators.clean import split_legal_form

ops = duckdb.sql(f"""
SELECT operator, COUNT(DISTINCT epr_number) AS permits,
       ROUND(SUM(tonnes_received)) AS tonnes_received
FROM {f} WHERE ewc_chapter NOT LIKE '19%'
GROUP BY operator ORDER BY tonnes_received DESC
""").df()

ops[["stem", "legal_form"]] = ops["operator"].apply(lambda n: pd.Series(split_legal_form(n)))
print(len(ops), ops["stem"].nunique())
print(ops["legal_form"].value_counts(dropna=False).head())
ops.head(15)

In [ ]:
# The 750 with no legal form: what are they?
# ops[ops["legal_form"] == ""].head(25)[["operator", "permits", "tonnes_received"]]

# # The two stem collisions
# ops[ops.duplicated("stem", keep=False)].sort_values("stem")

In [ ]:
duckdb.sql("COPY ops TO 'data/interim/operators_in_scope.parquet' (FORMAT PARQUET)")

In [ ]:
# ops[ops["legal_form"] == ""].head(25)[["operator", "permits", "tonnes_received"]]

In [ ]:
ops[ops["legal_form"] == ""][["operator", "permits", "tonnes_received", "stem"]].to_csv("~/Downloads/750records.csv", index=False)

In [ ]:
ops.head()

In [ ]:
# q(f"""SELECT operator, site_name, facility_type, ROUND(SUM(tonnes_received)) AS tonnes
# FROM {f}
# WHERE operator IN ('TAMWORTH BOROUGH COUNCIL','Roland Head','Jay Mould','Pamela O''Gara')
#   AND ewc_chapter NOT LIKE '19%'
# GROUP BY ALL ORDER BY tonnes DESC LIMIT 10""")

In [15]:
import duckdb, pandas as pd, sys
sys.path.append("src")
f = "'data/interim/waste_received_2025.parquet'"
r = "'data/interim/waste_removed_2025.parquet'"
q = lambda sql: duckdb.sql(sql).df()

In [16]:
r = "'data/interim/waste_removed_2025.parquet'"
display(q(f"SELECT ewc_chapter, ROUND(SUM(tonnes_removed)) AS tonnes FROM {r} GROUP BY 1 ORDER BY 1"))
display(q(f"SELECT fate, ROUND(SUM(tonnes_removed)) AS tonnes FROM {r} GROUP BY 1 ORDER BY 2 DESC"))
display(q(f"""SELECT destination_region, ROUND(SUM(tonnes_removed)) AS tonnes FROM {r}
WHERE facility_rpa = 'East Midlands' GROUP BY 1 ORDER BY 2 DESC LIMIT 5"""))

,ewc_chapter,tonnes
0,01 - MINE AND QUARRY WASTES,136115.0
1,02 - AGRICULTURE AND FOOD PROCESSING WASTES,934486.0
2,"03 - FURNITURE, PAPER AND CARDBOARD MANUFACTUR...",537155.0
3,"04 - LEATHER, FUR AND TEXTILE INDUSTRY WASTES",2870.0
4,"05 - PETROLEUM, GAS AND COAL PROCESSING WASTES",3129.0
5,06 - INORGANIC CHEMICAL PROCESS WASTE,23347.0
6,07 - ORGANIC CHEMICAL PROCESS WASTE,76231.0
7,"08 - PAINT, ADHESIVE, SEALANT AND INK MANUFACT...",60822.0
8,09 - PHOTOGRAPHIC INDUSTRY WASTES,638.0
9,10 - THERMAL PROCESSES WASTE,1367496.0


,fate,tonnes
0,Recovery,71769891.0
1,Incineration,16427571.0
2,Landfill,9828110.0
3,Treatment,8604470.0
4,Transfer (D),6253855.0
5,Other Fate,1336396.0
6,Long term storage,39573.0
7,None,114.0


,destination_region,tonnes
0,East Midlands,5782152.0
1,Yorks & Humber,1090500.0
2,West Midlands,954900.0
3,Outside UK,633799.0
4,East of England,557228.0
